In [3]:
import os
from dotenv import load_dotenv

# 아래에 코드를 작성하세요
load_dotenv()

True

In [4]:
SQUAD_ANSWER_FILE_PATH = "./data/ko_nia_normal_squad_all.json"
SQUAD_NOANSWER_FILE_PATH = "./data/ko_nia_noanswer_squad_all.json"

In [5]:
import os
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

from sklearn.model_selection import train_test_split

def read_squad(path):
    path = Path(path)
    with open(path, 'rb') as f:
        squad_dict = json.load(f)

    contexts = []
    questions = []
    answers = []
    group_numbers = []
    for group in squad_dict['data']:
        # if group['source'] != 2:
            # continue
        group_number = group['source']
        for passage in group['paragraphs']:
            context = passage['context']
            for qa in passage['qas']:
                question = qa['question']
                for answer in qa['answers']:
                    contexts.append(context)
                    questions.append(question)
                    answers.append(answer)
                    group_numbers.append(group_number)

    return contexts, questions, answers, group_numbers

total_contexts, total_questions, total_answers, group_numbers = read_squad(SQUAD_ANSWER_FILE_PATH)

In [6]:
def add_end_idx(answers, contexts):
    for answer, context in zip(answers, contexts):
        answer['text'] = answer['text'].rstrip()
        gold_text = answer['text']
        start_idx = answer['answer_start']
        end_idx = start_idx + len(gold_text)

        assert context[start_idx:end_idx] == gold_text, "end_index 계산에 에러가 있습니다."
        answer['answer_end'] = end_idx

add_end_idx(total_answers, total_contexts)

In [7]:
df = pd.DataFrame({'contexts': total_contexts, 'questions': total_questions, 'answers': total_answers, 'group_id': group_numbers})
df.head()

,contexts,questions,answers,group_id
0,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,서울과 충북 괴산에서 '국제청소년포럼'을 여는 곳은?,"{'answer_start': 0, 'text': '한국청소년단체협의회와 여성가족부...",5
1,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,'국제 청소년포럼'이 열리는 때는?,"{'answer_start': 19, 'text': '22일부터 28일', 'ans...",5
2,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,이번 포럼의 주제는?,"{'answer_start': 157, 'text': ''청소년과 뉴미디어'', '...",5
3,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,포럼은 어떻게 진행되는가?,"{'answer_start': 232, 'text': '기조강연을 시작으로 국가별 ...",5
4,[헤럴드POP=고승아 기자]그룹 구구단 샐리가 리듬체조에서 실수했다.15일 방송된 ...,아육대에서 리듬체조에 출전한 구구단의 멤버는?,"{'answer_start': 22, 'text': '샐리', 'answer_end...",7


In [8]:
# 각 group_id에서 상위 500개만 남긴다.
result_df = pd.DataFrame()

for group_id in df['group_id'].unique():
    subset = df[df['group_id'] == group_id].head(500)
    result_df = pd.concat([result_df, subset])

result_df = result_df.reset_index(drop=True)

# 결과 데이터프레임 확인
result_df['group_id'].value_counts()

group_id
5    500
7    500
4    500
3    500
2    500
6    500
9    500
1    500
8    500
Name: count, dtype: int64

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [13]:
# 단답을 배제하고, gpt가 긴 답변을 할 수 있도록 프롬프트 생성
system_prompt = """
당신은 본문으로부터 질문의 답변을 작성하는 언어모델입니다.
본문에 질문의 답이 없는 경우 찾을 수 없다고 답변하세요. 너무 짧게 답변하지 마세요. 답변은 세 줄 이상 작성하세요.
"""

In [11]:
# 데이터 프레임에 있는 본문(질문과 답변을 이용해 user prompt를 생성
user_prompt = []
for c, q in zip(result_df['contexts'].to_list(), result_df['questions'].to_list()):
  user_prompt.append('본문: ' + c + '\n' + '질문: ' + q + '\n답변:')

In [15]:
response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_prompt)
])

response

RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4o in organization org-l8bHkJoBAxKDae5VEQxZgO9g on tokens per min (TPM): Limit 2000000, Requested 2577109. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}